<a href="https://colab.research.google.com/github/Interuniversalism/Evaluating-Socio-Ecological-Bias-in-Retrieval-Augmented-Generation/blob/main/Ready_for_github_RAG_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-core langchain-community langchain-ollama chromadb

In [ ]:
import os
import json
import time
import requests
from google.colab import drive
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document


##############################################
# ✅ Mount Google Drive
##############################################

#def mount_drive():
    #drive.mount('/content/drive')


##############################################
# ✅ Load JSON Chunks
##############################################

def load_json_files_from_folder(folder_path):
    all_chunks = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as file:
                try:
                    chunks = json.load(file)
                    for c in chunks:
                        if isinstance(c, str) and len(c.strip()) > 50:
                            all_chunks.append((c, filename))
                except json.JSONDecodeError:
                    print(f"❌ JSON Error: {filename}")
    return all_chunks


##############################################
# ✅ Ultra-Safe Ollama Embedding
##############################################

class LlamaEmbeddings:
    def __init__(
        self,
        base_url="http://localhost:11434",
        model="nomic-embed-text",
        max_chars=1400,
        timeout=120,
        retries=4,
        embedding_dimension=768 # nomic-embed-text provides 768-dim embeddings
    ):
        self.base_url = base_url
        self.model = model
        self.max_chars = max_chars
        self.timeout = timeout
        self.retries = retries
        self.embedding_dimension = embedding_dimension

    def embed_query(self, text):
        text = str(text).strip()[:self.max_chars]

        if len(text) < 30:
            # Return a zero vector for very short texts instead of None
            return [0.0] * self.embedding_dimension

        for i in range(self.retries):
            try:
                r = requests.post(
                    f"{self.base_url}/api/embeddings",
                    json={"model": self.model, "prompt": text},
                    timeout=self.timeout
                )
                r.raise_for_status()
                emb = r.json().get("embedding")

                if isinstance(emb, list) and len(emb) == self.embedding_dimension:
                    return emb
                elif isinstance(emb, list) and len(emb) > 10: # Fallback for unknown dimensions
                    print(f"⚠️ Warning: Embedding dimension mismatch. Expected {self.embedding_dimension}, got {len(emb)}. Using provided embedding.")
                    return emb

            except Exception as e:
                print(f"⚠️ Retry {i+1}: {e}")
                torch.cuda.empty_cache()
                time.sleep(2)

        print("❌ Embedding failed. Returning zero vector.")
        # Return a zero vector if embedding consistently fails
        return [0.0] * self.embedding_dimension


##############################################
# ✅ Add safe documents only
##############################################

def add_documents_to_chromadb(vectorstore, chunks_with_meta, embedder):
    ok, bad = 0, 0
    documents = []
    embeddings = []

    for text, meta in chunks_with_meta:
        emb = embedder.embed_query(text)

        # Only add documents with valid embeddings (not a zero vector if that signals failure)
        # For this context, we will consider a zero vector as a valid but failed embedding to store
        # You might want to skip these documents if you prefer not to store failed embeddings
        if emb and any(e != 0.0 for e in emb): # Check if it's not a pure zero vector
            documents.append(Document(page_content=text, metadata={"source_file": meta}))
            embeddings.append(emb)
            ok += 1
        else:
            bad += 1

        if ok % 20 == 0 and ok > 0:
            print(f"✅ Stored: {ok} | ❌ Skipped: {bad}")

    if not documents:
        print("❌ Nothing to store with valid embeddings.")
        return

    vectorstore._collection.add(
        documents=[doc.page_content for doc in documents],
        metadatas=[doc.metadata for doc in documents],
        embeddings=embeddings,
        ids=[str(i) for i in range(len(embeddings))]
    )

    vectorstore.persist()

    print(f"\n✅ FINAL RESULT: {ok} stored | {bad} skipped")


##############################################
# ✅ Retrieve safely
##############################################

def retrieve_all_documents(vectorstore):
    retriever = vectorstore.as_retriever()
    # The query 'test' will now also be embedded by LlamaEmbeddings, returning a zero vector if failed.
    return retriever.invoke("test")


##############################################
# ✅ MAIN
##############################################

def main():
    global vectorstore # Declare vectorstore as global

    #mount_drive()

    folder_path = "...."
    persist_path = "...."

    print("📂 Loading chunks...")
    chunks = load_json_files_from_folder(folder_path)
    print(f"✅ Loaded {len(chunks)} valid chunks.")

    embedder = LlamaEmbeddings(
        model="nomic-embed-text",
        max_chars=1400,
        embedding_dimension=768
    )

    # Fix: Initialize Chroma with the correct embedding function
    vectorstore = Chroma(
        persist_directory=persist_path,
        embedding_function=embedder # Pass the embedder here
    )

    print("🚀 Embedding & Saving...")
    add_documents_to_chromadb(vectorstore, chunks, embedder)

    print("🔍 Testing retrieval...")
    docs = retrieve_all_documents(vectorstore)

    for i, d in enumerate(docs[:5], 1):
        print(f"{i}. {d.page_content[:120]}")

    print("✅ All Done.")


##############################################

if __name__ == "__main__":
    main()

In [ ]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama

ollama = Ollama(
    base_url='http://localhost:11434',
    model="mistral",
    num_ctx=8094
)

ollama_embeddings = LlamaEmbeddings(
        model="nomic-embed-text"
    )

In [ ]:
import itertools
import random

# ✅ تعریف مؤلفه‌های پرامپ برای ترکیب
verbs = ["Identify", "Extract", "Map", "Analyze"]
frameworks = ["DPSIR", "Drivers–Pressures–States–Impacts–Responses"]
scopes = [
    "agricultural resilience in Henan Province",
    "resilience of agriculture in henan",
    "vulnerability of agriculture in Central China"
]
variable_types = [
    "ecological and social variables"
]
linkages = [
    "Make explicit the linkages",
    "Explain how categories relate",
    "Describe causal chains between components"
]

# ✅ قالب کلی برای پرامپ
template = """the five DPSIR(Driver, Pressure, State, Impact, Response) framework definition in Agricultural resilience context as below:
Driver: Broad socio-economic or environmental forces (e.g., policy reforms, climate variability, demographic trends) shape agricultural systems over time.
Pressure: Direct human-induced or natural stressors on agroecosystems Based on extracted Drivers (e.g., land degradation, overuse of agrochemicals, irrigation intensity).
State: Current condition of the agroecosystem and socio-economic attributes those Pressurs lead (e.g., soil fertility, water availability, biodiversity, farmer income, land tenure).
Impact: Consequences of changes in the state (e.g., reduced crop yields, increased poverty, food insecurity, ecological degradation).
Response: Actions or interventions (e.g., adaptive farming practices, subsidies, training programs, policy changes) to mitigate pressures or enhance resilience.
{} from any given data or document the {} that correspond to each of the {} categories, specifically in relation to {}. {} between categories (e.g., how a Driver leads to a Pressure)."""

# ✅ تولید ۵۰ پرامپ تصادفی از بین ترکیب‌ها
all_combinations = list(itertools.product(verbs, variable_types, frameworks, scopes, linkages))
selected_prompts = random.sample(all_combinations, 20)
final_prompts = [template.format(*combo) for combo in selected_prompts]

# ✅ چاپ پرامپ‌ها
for i, prompt in enumerate(final_prompts):
    print(f"🟢 Prompt {i+1}:\n{prompt}\n")


🟢 Prompt 1:
the five DPSIR(Driver, Pressure, State, Impact, Response) framework definition in Agricultural resilience context as below:
Driver: Broad socio-economic or environmental forces (e.g., policy reforms, climate variability, demographic trends) that shape agricultural systems over time.
Pressure: Direct human-induced or natural stressors on agroecosystems (e.g., land degradation, overuse of agrochemicals, irrigation intensity).
State: Current condition of the agroecosystem and socio-economic attributes (e.g., soil fertility, water availability, biodiversity, farmer income, land tenure).
Impact: Consequences of changes in the state (e.g., reduced crop yields, increased poverty, food insecurity, ecological degradation).
Response: Actions or interventions (e.g., adaptive farming practices, subsidies, training programs, policy changes) to mitigate pressures or enhance resilience.
Analyze from any given data or document the ecological and social variables that correspond to each of 

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# --------------------
# 🔹 Cosine similarity
# --------------------
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def compute_cosine(a, b):
    return float(cosine_similarity([a], [b])[0][0])


# --------------------
# 🔹 RAG Function
# --------------------
def simple_rag_query_with_similarity(question, vectorstore, llm, embed_model, k=4):
    global answer, docs, cosine_scores, full_context
    try:
        retriever = vectorstore.as_retriever(search_kwargs={"k": k})
        docs = retriever.invoke(question)
        print("Retrieved docs:", len(docs))

        for i, d in enumerate(docs):
            print(f"Doc {i+1} length:", len(d.page_content))

        q_emb = np.array(embed_model.embed_query(question))
        cosine_scores = []
        context_parts = []

        for i, doc in enumerate(docs):
            chunk = doc.page_content
            c_emb = np.array(embed_model.embed_query(chunk))
            sim = cosine_sim(q_emb, c_emb)

            cosine_scores.append(sim)
            context_parts.append(f"[Document {i+1} | Cosine={sim:.4f}]: {chunk}")

        full_context = "\n\n".join(context_parts)

        prompt = f"""
You are an expert analyst. Based on the documents, answer the question thoroughly.

QUESTION: {question}

DOCUMENTS:
{full_context}

Please answer ONLY based on these documents.
"""

        answer = llm.invoke(prompt)

        return {
            "answer": answer,
            "docs": docs,
            "cosine_scores": cosine_scores,
            "context": full_context
        }

    except Exception as e:
        return {"error": str(e)}


# --------------------
# 🔹 Compute similarities
# --------------------
def compute_answer_similarities(answer, question, context, embedding_model):
    answer_vec = embedding_model.embed_query(answer)
    question_vec = embedding_model.embed_query(question)
    context_vec = embedding_model.embed_query(context)

    return {
        "answer_question_similarity": compute_cosine(answer_vec, question_vec),
        "answer_doc_similarity": compute_cosine(answer_vec, context_vec)
    }


In [ ]:
import csv

output_file = " deep_rag_evaluation_results_social.csv"

with open(output_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)

    writer.writerow([
        "question",
        "answer",
        "answer_question_similarity",
        "answer_doc_similarity",
        "top_k_docs"
    ])

    # ----- RUN the RAG-----
    for idx, question in enumerate(final_prompts):
        print(f"\n========================")
        print(f"🔄 Running question {idx+1} / {len(final_prompts)}")
        print("========================")

        result = simple_rag_query_with_similarity(
            question=question,
            vectorstore=vectorstore,
            llm=ollama,
            embed_model=ollama_embeddings,
            k=4
        )

        if "error" in result:
            print("❌ Error:", result["error"])
            continue

        answer = result["answer"]
        docs = result["docs"]
        context = result["context"]

        scores = compute_answer_similarities(
            answer,
            question,
            context,
            ollama_embeddings
        )

        top_docs = " ||| ".join([d.page_content[:1000] for d in docs])

        writer.writerow([
            question,
            answer,
            scores["answer_question_similarity"],
            scores["answer_doc_similarity"],
            top_docs
        ])

print("🎉 DONE — CSV created:", output_file)
